In [14]:
# Gradient Descent for Linear Regression (Multiple Variables)
# Model: ŷ = X @ w + b
# Cost:  J = (1/m) * ||ŷ - y||²
import numpy as np

np.set_printoptions(
    suppress=True,
    formatter={"float_kind": "{:0.6f}".format},
)

# Linear Regression: Gradient Descent for Multiple Variables

## Problem Formulation

Given training data with $m$ samples and $n$ features:
- $\mathbf{X} \in \mathbb{R}^{m \times n}$: feature matrix
- $\mathbf{y} \in \mathbb{R}^m$: target vector

Learn parameters:
- $\mathbf{w} \in \mathbb{R}^n$: weight vector (one per feature)
- $b \in \mathbb{R}$: bias/intercept term

## Model

Linear hypothesis:
$$\hat{\mathbf{y}} = \mathbf{X}\mathbf{w} + b$$

where $\hat{\mathbf{y}} \in \mathbb{R}^m$ are predictions.

## Cost Function (Mean Squared Error)

$$\mathcal{J}(\mathbf{w}, b) = \frac{1}{m}\|\hat{\mathbf{y}} - \mathbf{y}\|^2 = \frac{1}{m}\|(\mathbf{X}\mathbf{w} + b) - \mathbf{y}\|^2$$

## Gradient Descent Algorithm

At each epoch, update parameters opposite to gradient direction:

1. **Forward pass:** $\hat{\mathbf{y}} = \mathbf{X}\mathbf{w} + b$
2. **Compute error:** $\boldsymbol{\delta} = \hat{\mathbf{y}} - \mathbf{y}$
3. **Partial Derivatives:**
   - $\frac{\partial \mathcal{J}}{\partial \mathbf{w}} = \frac{2}{m}\mathbf{X}^T\boldsymbol{\delta}$
   - $\frac{\partial \mathcal{J}}{\partial b} = \frac{2}{m}\sum_i \delta_i$
4. **Update** (with learning rate $\alpha > 0$):
   - $\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial \mathcal{J}}{\partial \mathbf{w}}$
   - $b \leftarrow b - \alpha \frac{\partial \mathcal{J}}{\partial b}$

**Key insight:** $\mathbf{X}^T\boldsymbol{\delta}$ has shape $(n, m) \times (m,) = (n,)$ — one gradient per feature.

## Feature Normalization (Standardization)

When features have different scales, gradient descent becomes unstable and weights may explode to $\infty$ or $\text{NaN}$.

**Solution:** Standardize to zero mean and unit variance:
$$\mathbf{X}_{\text{norm}} = \frac{\mathbf{X} - \boldsymbol{\mu}}{\boldsymbol{\sigma}}$$

where $\boldsymbol{\mu}, \boldsymbol{\sigma}$ are computed per feature (column-wise).

**Why it works:** Normalized features → well-conditioned gradients → stable convergence.

In [15]:
def normalize(X, y=None):
    """
    Normalize features to have mean=0 and std=1.
    
    This is crucial for stable gradient descent when features have different scales.
    Prevents numerical instability (weights exploding to infinity).
    
    Returns:
        X_norm, y_norm, X_mean, X_std, y_mean, y_std
    """
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)
    X_norm = (X - X_mean) / X_std
    
    y_mean = np.mean(y) if y is not None else 0
    y_std = np.std(y) if y is not None else 1
    y_norm = (y - y_mean) / y_std if y is not None else None
    
    return X_norm, y_norm, X_mean, X_std, y_mean, y_std

In [16]:
def denormalize(X_norm, y_norm, X_mean, X_std, y_mean, y_std):
    """
    Reverse normalization to get back to original scale.
    
    Given normalized data and the statistics from normalize(),
    reconstruct the original data.
    
    Args:
        X_norm: Normalized features
        y_norm: Normalized target
        X_mean, X_std: Feature statistics from normalize()
        y_mean, y_std: Target statistics from normalize()
    
    Returns:
        X, y in original scale
    """
    X = X_norm * X_std + X_mean
    y = y_norm * y_std + y_mean
    return X, y

In [ ]:
def compute_mse(X, y, w, b):
    """
    Compute Mean Squared Error (Cost Function): J = (1/m) ||ŷ - y||²
    
    Args:
        X: feature matrix, shape (m, n)
        y: target vector, shape (m,)
        w: weight vector, shape (n,)
        b: bias scalar
    
    Returns:
        Cost value (scalar)
    """
    y_hat = X @ w + b
    return np.mean((y_hat - y) ** 2)

    
def gradient_step(X, y, w, b, lr):
    """
    One gradient descent step: θ ← θ - α(∂J/∂θ)
    
    Args:
        X: feature matrix, shape (m, n)
        y: target vector, shape (m,)
        w: weight vector, shape (n,)
        b: bias scalar
        lr: learning rate α > 0
    
    Returns:
        updated w, b
    """
    m = X.shape[0]
    
    # Forward pass: ŷ = Xw + b
    y_hat = X @ w + b
    
    # Error term δ = ŷ - y (shape: (m,))
    error = y_hat - y

    # Partial derivative w.r.t. weights: ∂J/∂w = (2/m) X^T δ (shape: (n,))
    # Matrix multiplication: (n,m) @ (m,) → (n,)
    dw = (2 / m) * (X.T @ error)
    
    # Partial derivative w.r.t. bias: ∂J/∂b = (2/m) Σ δ_i (scalar)
    db = (2 / m) * np.sum(error)

    # Update: θ ← θ - α(∂J/∂θ)
    w = w - lr * dw
    b = b - lr * db
    
    return w, b

In [ ]:
def gradient_descent(X, y, lr=0.01, epochs=1000, debug_every=100):
    """
    Batch gradient descent for linear regression: minimize J(w,b).
    
    Args:
        X: feature matrix, shape (m, n)
        y: target vector, shape (m,)
        lr: learning rate α (default 0.01)
        epochs: number of iterations (default 1000)
        debug_every: print loss every N epochs
    
    Returns:
        w, b: learned parameters
    """
    m, n = X.shape
    
    # Initialize: w ← 0, b ← 0
    w = np.zeros(n)
    b = 0.0

    for epoch in range(epochs):
        w, b = gradient_step(X, y, w, b, lr)

        # Monitor loss during training
        if epoch % debug_every == 0 or epoch == epochs - 1:
            mse = compute_mse(X, y, w, b)
            print(f"epoch={epoch:5d} | loss={mse:12.6f} | w={w} | b={b:.6f}")

    return w, b

In [19]:
# Data: X ∈ R^(5x3), y ∈ R^5
X = np.array([
    [120, 3, 10],   # 120 m², 3 beds, 10 yrs
    [150, 4, 5],    # 150 m², 4 beds, 5 yrs
    [90,  2, 20],   # 90 m², 2 beds, 20 yrs
    [200, 5, 3],    # 200 m², 5 beds, 3 yrs
    [110, 3, 15]    # 110 m², 3 beds, 15 yrs
], dtype=float)

# Target: house prices (in 100k units)
y = np.array([300, 420, 210, 560, 270], dtype=float)

print(f"Data shape: X ∈ R^{X.shape}, y ∈ R^{y.shape}")

# Normalize to prevent numerical instability
X_norm, y_norm, X_mean, X_std, y_mean, y_std = normalize(X, y)

print(f"\nFeature statistics (original scale):")
print(f"  μ = {X_mean},  σ = {X_std}")
print(f"Target statistics: μ={y_mean:.2f}, σ={y_std:.2f}")

# Initialize parameters: w ← 0 ∈ R^n, b ← 0
m, n = X_norm.shape
w = np.zeros(n)
b = 0.0

# Hyperparameters
lr = 0.01          # learning rate α
epochs = 1000

# Train: minimize J(w,b) via gradient descent
print(f"\nTraining (m={m} samples, n={n} features, α={lr}, T={epochs} epochs):\n")
w, b = gradient_descent(X_norm, y_norm, lr, epochs, debug_every=100)

print(f"\n✓ Learned parameters (in normalized space):")
print(f"  w = {w}")
print(f"  b = {b:.6f}")

Data shape: X ∈ R^(5, 3), y ∈ R^(5,)

Feature statistics (original scale):
  μ = [134.000000 3.400000 10.600000],  σ = [38.262253 1.019804 6.280127]
Target statistics: μ=352.00, σ=124.48

Training (m=5 samples, n=3 features, α=0.01, T=1000 epochs):

epoch=    0 | loss=    0.890133 | w=[0.019954 0.019725 -0.018553] | b=0.000000
epoch=  100 | loss=    0.018608 | w=[0.381939 0.346494 -0.273591] | b=0.000000
epoch=  200 | loss=    0.014832 | w=[0.422629 0.352968 -0.228115] | b=0.000000
epoch=  300 | loss=    0.012169 | w=[0.456989 0.356323 -0.189813] | b=0.000000
epoch=  400 | loss=    0.010277 | w=[0.486711 0.357788 -0.158147] | b=0.000000
epoch=  500 | loss=    0.008925 | w=[0.512539 0.357712 -0.132012] | b=0.000000
epoch=  600 | loss=    0.007949 | w=[0.535094 0.356383 -0.110481] | b=-0.000000
epoch=  700 | loss=    0.007238 | w=[0.554892 0.354041 -0.092785] | b=-0.000000
epoch=  800 | loss=    0.006711 | w=[0.572366 0.350886 -0.078281] | b=-0.000000
epoch=  900 | loss=    0.006315 | w=

## Example 1: House Price Prediction

Predict house price from 3 features: `[size (m²), bedrooms, age (years)]`

### Make Predictions

To predict on new data, we must:
1. Normalize inputs using training statistics: $\mathbf{x}_{\text{norm}} = \frac{\mathbf{x} - \boldsymbol{\mu}}{\boldsymbol{\sigma}}$
2. Compute prediction: $\hat{y}_{\text{norm}} = \mathbf{x}_{\text{norm}}^T\mathbf{w} + b$
3. Denormalize to original scale: $\hat{y} = \hat{y}_{\text{norm}} \cdot \sigma_y + \mu_y$

In [21]:
# Predict on new house
new_house = np.array([140, 3, 8])  # 140 m², 3 beds, 8 yrs

# Step 1: Normalize using training statistics
new_house_norm = (new_house - X_mean) / X_std

# Step 2: Predict in normalized space
y_pred_norm = new_house_norm @ w + b

# Step 3: Denormalize to original scale
y_pred = y_pred_norm * y_std + y_mean

print(f"New house: {new_house}")
print(f"Predicted price: ${y_pred * 1000:,.0f}")

New house: [140   3   8]
Predicted price: $349,937


## Summary & Key Takeaways

| Concept | Formula | Shape |
|---------|---------|-------|
| **Model** | $\hat{\mathbf{y}} = \mathbf{X}\mathbf{w} + b$ | $(m,) = (m,n) @ (n,) + ()$ |
| **Cost** | $\mathcal{J} = \frac{1}{m}\|\hat{\mathbf{y}} - \mathbf{y}\|^2$ | scalar |
| **∂J/∂w** | $\frac{\partial \mathcal{J}}{\partial \mathbf{w}} = \frac{2}{m}\mathbf{X}^T(\hat{\mathbf{y}} - \mathbf{y})$ | $(n,)$ |
| **∂J/∂b** | $\frac{\partial \mathcal{J}}{\partial b} = \frac{2}{m}\sum(\hat{\mathbf{y}} - \mathbf{y})$ | scalar |
| **Update (w)** | $\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial \mathcal{J}}{\partial \mathbf{w}}$ | $(n,)$ |
| **Update (b)** | $b \leftarrow b - \alpha \frac{\partial \mathcal{J}}{\partial b}$ | scalar |

**Best Practices:**
- ✓ Always normalize features
- ✓ Initialize weights to zero or small random values
- ✓ Use appropriate learning rate $\alpha$ (0.001—0.1 typically)
- ✓ Monitor cost function during training
- ✓ Use same normalization stats for train and test